# Photos Library Messages Attachment Recovery

This notebook is a read-only exploration notebook for recovering original files behind Photos **Shared with You / syndication** placeholder records.

Goal:

1. Find Photos records where `syndicated == True`, `saved_to_library == False`, `path is None`, and `ismissing == True`.
2. Inspect their derivative previews and metadata.
3. Build a read-only inventory of macOS Messages attachments from `~/Library/Messages/chat.db` and `~/Library/Messages/Attachments`.
4. Match Photos syndicated placeholders to possible Messages attachment originals.
5. Produce review tables only. This notebook does not delete, import, or modify Photos or Messages data.


In [ ]:
# ============================================================
# Section 1: Imports and settings
# ============================================================

from pathlib import Path
from datetime import datetime, timezone, timedelta
from collections import Counter, defaultdict
import csv
import hashlib
import json
import os
import shutil
import sqlite3
import subprocess
import time

import osxphotos

try:
    from PIL import Image
except Exception:
    Image = None

from explorephotoslibrary import load_json_file, save_json_file, choose_photos_library_path

PROJECT_ROOT = Path.cwd()
LOCAL_CONFIG_DIR = PROJECT_ROOT / "data" / "local_config"
OUTPUT_DIR = PROJECT_ROOT / "data" / "messages_attachment_recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST2_LIBRARY_PATHS_CONFIG_PATH = LOCAL_CONFIG_DIR / "test2_library_paths.json"
CURRENT_LIBRARY_KEY = "current_default"

MESSAGES_DIR = Path.home() / "Library" / "Messages"
MESSAGES_DB_PATH = MESSAGES_DIR / "chat.db"
MESSAGES_ATTACHMENTS_ROOT = MESSAGES_DIR / "Attachments"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MESSAGES_DB_PATH:", MESSAGES_DB_PATH)
print("MESSAGES_ATTACHMENTS_ROOT:", MESSAGES_ATTACHMENTS_ROOT)
print("PIL available:", Image is not None)


In [ ]:
# ============================================================
# Section 2: Resolve CURRENT Photos Library path
# ============================================================

library_paths = load_json_file(TEST2_LIBRARY_PATHS_CONFIG_PATH, default={}) or {}

if CURRENT_LIBRARY_KEY in library_paths:
    current_photos_library_path = Path(library_paths[CURRENT_LIBRARY_KEY])
    print("Using saved Photos Library path for:", CURRENT_LIBRARY_KEY)
else:
    print("No saved current_default path found; opening picker.")
    current_photos_library_path = Path(
        choose_photos_library_path(
            initial_dir=Path.home() / "Pictures",
            prompt="Select CURRENT default Photos Library"
        )
    )
    library_paths[CURRENT_LIBRARY_KEY] = str(current_photos_library_path)
    save_json_file(TEST2_LIBRARY_PATHS_CONFIG_PATH, library_paths)

print("current_photos_library_path:", current_photos_library_path)
if not current_photos_library_path.exists():
    raise FileNotFoundError(current_photos_library_path)


In [ ]:
# ============================================================
# Section 3: Collect Photos syndicated placeholder records
# ============================================================

def safe_iso(value):
    if value is None:
        return None
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)


def path_list(value):
    if not value:
        return []
    return [str(item) for item in value]


def get_image_dimensions(path):
    if path is None or Image is None:
        return None, None
    try:
        with Image.open(path) as img:
            return img.size
    except Exception:
        return None, None


def sha256_file(path, chunk_size=1024 * 1024):
    if path is None:
        return None
    path = Path(path)
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

photosdb = osxphotos.PhotosDB(dbfile=str(current_photos_library_path))
photos = photosdb.photos()

syndicated_records = []

for photo in photos:
    syndicated = bool(getattr(photo, "syndicated", False))
    saved_to_library = bool(getattr(photo, "saved_to_library", False))
    ismissing = bool(getattr(photo, "ismissing", False))
    path = str(photo.path) if photo.path else None
    original_filesize = getattr(photo, "original_filesize", None)
    derivatives = path_list(getattr(photo, "path_derivatives", []) or [])

    # This is intentionally the same condition that exposed the 38 records:
    # Shared-with-You / syndication placeholder, not solidified as a normal original.
    if not (syndicated and not saved_to_library and path is None):
        continue

    derivative_records = []
    for derivative_path in derivatives:
        p = Path(derivative_path)
        width, height = get_image_dimensions(p)
        derivative_records.append({
            "path": str(p),
            "exists": p.exists(),
            "size_bytes": p.stat().st_size if p.exists() else None,
            "width": width,
            "height": height,
            "sha256": sha256_file(p) if p.exists() else None,
        })

    syndicated_records.append({
        "uuid": photo.uuid,
        "original_filename": photo.original_filename,
        "filename": photo.filename,
        "date": safe_iso(photo.date),
        "date_added": safe_iso(photo.date_added),
        "date_original": safe_iso(getattr(photo, "date_original", None)),
        "syndicated": syndicated,
        "saved_to_library": saved_to_library,
        "ismissing": ismissing,
        "path": path,
        "original_filesize": original_filesize,
        "uti": getattr(photo, "uti", None),
        "uti_original": getattr(photo, "uti_original", None),
        "is_movie": bool(photo.ismovie),
        "live_photo": bool(getattr(photo, "live_photo", False)),
        "width": getattr(photo, "width", None),
        "height": getattr(photo, "height", None),
        "original_width": getattr(photo, "original_width", None),
        "original_height": getattr(photo, "original_height", None),
        "location": getattr(photo, "location", None),
        "latitude": getattr(photo, "latitude", None),
        "longitude": getattr(photo, "longitude", None),
        "ai_caption": getattr(photo, "ai_caption", None),
        "albums": list(photo.albums or []),
        "keywords": list(photo.keywords or []),
        "description": photo.description,
        "favorite": bool(photo.favorite),
        "hidden": getattr(photo, "hidden", None),
        "derivatives": derivative_records,
    })

syndicated_records = sorted(
    syndicated_records,
    key=lambda r: (r["date"] or "", r["original_filename"] or "", r["uuid"] or "")
)

print("syndicated placeholder records:", len(syndicated_records))
print()
print("status counts:")
for field in ["syndicated", "saved_to_library", "ismissing", "path", "original_filesize"]:
    c = Counter()
    for r in syndicated_records:
        v = r[field]
        if field == "path":
            v = "PATH_NONE" if v is None else "PATH_PRESENT"
        if field == "original_filesize":
            v = "ZERO" if v == 0 else v
        c[v] += 1
    print(field, dict(c))

print()
for i, r in enumerate(syndicated_records[:20], start=1):
    print(f"{i:02d}. {r['original_filename']}  {r['date']}  {r['uuid']}  derivatives={len(r['derivatives'])}")


In [ ]:
# ============================================================
# Section 4: Save Photos syndicated placeholder inventory
# ============================================================

photos_syndicated_json_path = OUTPUT_DIR / "photos_syndicated_placeholders.json"
photos_syndicated_tsv_path = OUTPUT_DIR / "photos_syndicated_placeholders.tsv"

with photos_syndicated_json_path.open("w", encoding="utf-8") as f:
    json.dump(syndicated_records, f, ensure_ascii=False, indent=2, default=str)

fields = [
    "uuid", "original_filename", "filename", "date", "date_added", "date_original",
    "syndicated", "saved_to_library", "ismissing", "path", "original_filesize",
    "uti", "uti_original", "is_movie", "live_photo", "width", "height",
    "original_width", "original_height", "latitude", "longitude", "ai_caption",
    "derivative_count", "first_derivative_path", "first_derivative_size_bytes",
    "first_derivative_width", "first_derivative_height", "first_derivative_sha256",
]

with photos_syndicated_tsv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields, delimiter="	", extrasaction="ignore")
    writer.writeheader()
    for r in syndicated_records:
        d0 = r["derivatives"][0] if r["derivatives"] else {}
        row = dict(r)
        row["derivative_count"] = len(r["derivatives"])
        row["first_derivative_path"] = d0.get("path")
        row["first_derivative_size_bytes"] = d0.get("size_bytes")
        row["first_derivative_width"] = d0.get("width")
        row["first_derivative_height"] = d0.get("height")
        row["first_derivative_sha256"] = d0.get("sha256")
        writer.writerow(row)

print("Wrote:")
print(photos_syndicated_json_path)
print(photos_syndicated_tsv_path)


## Messages database snapshot

The next cells make a local read-only copy of `~/Library/Messages/chat.db` into the project output folder. This avoids reading a live SQLite database while Messages is using it. It does **not** modify the original Messages database.

macOS may ask for Full Disk Access / privacy permission when reading `~/Library/Messages`.


In [ ]:
# ============================================================
# Section 5: Copy Messages database snapshot read-only
# ============================================================

if not MESSAGES_DB_PATH.exists():
    raise FileNotFoundError(
        f"Messages database not found: {MESSAGES_DB_PATH}
"
        "On macOS you may need Full Disk Access for Terminal / Python / Jupyter."
    )

snapshot_dir = OUTPUT_DIR / "messages_db_snapshot"
snapshot_dir.mkdir(parents=True, exist_ok=True)

snapshot_db_path = snapshot_dir / "chat.db"
shutil.copy2(MESSAGES_DB_PATH, snapshot_db_path)

# Copy WAL/SHM if present. The main DB copy is often enough, but copying these
# improves consistency when Messages has recent uncheckpointed writes.
for suffix in ["-wal", "-shm"]:
    src = Path(str(MESSAGES_DB_PATH) + suffix)
    if src.exists():
        shutil.copy2(src, Path(str(snapshot_db_path) + suffix))

print("snapshot_db_path:", snapshot_db_path)
print("snapshot exists:", snapshot_db_path.exists(), "size:", snapshot_db_path.stat().st_size)


In [ ]:
# ============================================================
# Section 6: Inspect Messages database schema
# ============================================================

conn = sqlite3.connect(f"file:{snapshot_db_path}?mode=ro", uri=True)
conn.row_factory = sqlite3.Row

schema_rows = conn.execute(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY name"
).fetchall()

print("Tables/views:")
for row in schema_rows:
    print(row["type"], row["name"])

print()
for table in ["attachment", "message", "message_attachment_join", "handle", "chat", "chat_message_join"]:
    try:
        cols = conn.execute(f"PRAGMA table_info({table})").fetchall()
    except Exception as e:
        print(table, "ERROR", e)
        continue
    print("-" * 100)
    print(table)
    for col in cols:
        print(" ", col[1], col[2])


In [ ]:
# ============================================================
# Section 7: Build Messages attachment inventory
# ============================================================

def table_columns(conn, table):
    return [row[1] for row in conn.execute(f"PRAGMA table_info({table})").fetchall()]


def resolve_messages_attachment_path(filename):
    if not filename:
        return None

    text = str(filename)

    # Common Messages DB style: ~/Library/Messages/Attachments/...
    if text.startswith("~/"):
        p = Path.home() / text[2:]
    else:
        p = Path(text)

    if p.exists():
        return p

    # Some records may store only relative paths under Attachments.
    p2 = MESSAGES_ATTACHMENTS_ROOT / text
    if p2.exists():
        return p2

    return p


def apple_absolute_time_to_iso(value):
    # Handles common Apple timestamp forms:
    # - seconds since 2001-01-01
    # - nanoseconds since 2001-01-01
    if value is None:
        return None
    try:
        v = float(value)
    except Exception:
        return str(value)

    # Nanoseconds form is usually very large.
    if abs(v) > 10_000_000_000:
        v = v / 1_000_000_000

    apple_epoch = datetime(2001, 1, 1, tzinfo=timezone.utc)
    try:
        return (apple_epoch + timedelta(seconds=v)).astimezone().isoformat()
    except Exception:
        return str(value)

attachment_cols = table_columns(conn, "attachment")
print("attachment columns:", attachment_cols)

rows = conn.execute("SELECT * FROM attachment").fetchall()
messages_attachments = []

for row in rows:
    record = {col: row[col] for col in attachment_cols}

    attachment_path = resolve_messages_attachment_path(record.get("filename"))
    path_exists = bool(attachment_path and attachment_path.exists())
    path_size = attachment_path.stat().st_size if path_exists else None

    width = height = None
    if path_exists and Image is not None:
        try:
            with Image.open(attachment_path) as img:
                width, height = img.size
        except Exception:
            pass

    messages_attachments.append({
        "attachment_rowid": record.get("ROWID") or record.get("rowid"),
        "guid": record.get("guid"),
        "created_date_raw": record.get("created_date"),
        "created_date_iso": apple_absolute_time_to_iso(record.get("created_date")),
        "start_date_raw": record.get("start_date"),
        "start_date_iso": apple_absolute_time_to_iso(record.get("start_date")),
        "filename": record.get("filename"),
        "transfer_name": record.get("transfer_name"),
        "mime_type": record.get("mime_type"),
        "uti": record.get("uti"),
        "total_bytes": record.get("total_bytes"),
        "resolved_path": str(attachment_path) if attachment_path else None,
        "path_exists": path_exists,
        "path_size_bytes": path_size,
        "width": width,
        "height": height,
    })

print("Messages attachment records:", len(messages_attachments))
print("path exists:", Counter(a["path_exists"] for a in messages_attachments))
print("mime types top 20:")
for mime, count in Counter(a.get("mime_type") for a in messages_attachments).most_common(20):
    print(" ", mime, count)


In [ ]:
# ============================================================
# Section 8: Save Messages attachment inventory
# ============================================================

messages_attachments_json_path = OUTPUT_DIR / "messages_attachments.json"
messages_attachments_tsv_path = OUTPUT_DIR / "messages_attachments.tsv"

with messages_attachments_json_path.open("w", encoding="utf-8") as f:
    json.dump(messages_attachments, f, ensure_ascii=False, indent=2, default=str)

attachment_fields = [
    "attachment_rowid", "guid", "created_date_raw", "created_date_iso", "start_date_raw", "start_date_iso",
    "filename", "transfer_name", "mime_type", "uti", "total_bytes", "resolved_path",
    "path_exists", "path_size_bytes", "width", "height",
]

with messages_attachments_tsv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=attachment_fields, delimiter="	", extrasaction="ignore")
    writer.writeheader()
    writer.writerows(messages_attachments)

print("Wrote:")
print(messages_attachments_json_path)
print(messages_attachments_tsv_path)


In [ ]:
# ============================================================
# Section 9: Candidate matching between Photos syndicated placeholders
#            and Messages attachments
# ============================================================

def basename_lower(value):
    if not value:
        return None
    return Path(str(value)).name.lower()


def stem_lower(value):
    b = basename_lower(value)
    if not b:
        return None
    return Path(b).stem.lower()


def parse_iso_datetime(value):
    if not value:
        return None
    try:
        return datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    except Exception:
        return None


def datetime_distance_seconds(a, b):
    da = parse_iso_datetime(a)
    db = parse_iso_datetime(b)
    if da is None or db is None:
        return None
    if da.tzinfo is None:
        da = da.replace(tzinfo=timezone.utc)
    if db.tzinfo is None:
        db = db.replace(tzinfo=timezone.utc)
    return abs((da - db).total_seconds())


def score_candidate(photo, attachment):
    score = 0
    reasons = []

    photo_name = photo.get("original_filename") or photo.get("filename")
    attachment_names = [attachment.get("transfer_name"), attachment.get("filename")]

    photo_base = basename_lower(photo_name)
    photo_stem = stem_lower(photo_name)
    attachment_bases = [basename_lower(v) for v in attachment_names if v]
    attachment_stems = [stem_lower(v) for v in attachment_names if v]

    if photo_base and photo_base in attachment_bases:
        score += 100
        reasons.append("exact basename match")
    elif photo_stem and photo_stem in attachment_stems:
        score += 80
        reasons.append("filename stem match")

    # Dimensions can match the original or rendered orientation.
    photo_dims = {
        (photo.get("width"), photo.get("height")),
        (photo.get("height"), photo.get("width")),
        (photo.get("original_width"), photo.get("original_height")),
        (photo.get("original_height"), photo.get("original_width")),
    }
    attachment_dim = (attachment.get("width"), attachment.get("height"))

    if attachment_dim in photo_dims and attachment_dim != (None, None):
        score += 50
        reasons.append("dimension match")

    if attachment.get("path_exists"):
        score += 10
        reasons.append("attachment file exists")

    # Message attachment date may not equal photo capture date, so this is weak.
    for date_field in ["created_date_iso", "start_date_iso"]:
        distance = datetime_distance_seconds(photo.get("date"), attachment.get(date_field))
        if distance is None:
            continue
        if distance <= 60:
            score += 30
            reasons.append(f"{date_field} within 60s")
            break
        if distance <= 3600:
            score += 15
            reasons.append(f"{date_field} within 1h")
            break
        if distance <= 86400:
            score += 5
            reasons.append(f"{date_field} within 1d")
            break

    return score, reasons

candidates = []

for photo in syndicated_records:
    scored = []
    for attachment in messages_attachments:
        score, reasons = score_candidate(photo, attachment)
        if score <= 0:
            continue
        scored.append((score, reasons, attachment))

    scored.sort(key=lambda item: item[0], reverse=True)

    for rank, (score, reasons, attachment) in enumerate(scored[:20], start=1):
        candidates.append({
            "photo_uuid": photo.get("uuid"),
            "photo_original_filename": photo.get("original_filename"),
            "photo_date": photo.get("date"),
            "photo_width": photo.get("width"),
            "photo_height": photo.get("height"),
            "photo_original_width": photo.get("original_width"),
            "photo_original_height": photo.get("original_height"),
            "rank": rank,
            "score": score,
            "reasons": " | ".join(reasons),
            "attachment_rowid": attachment.get("attachment_rowid"),
            "attachment_guid": attachment.get("guid"),
            "attachment_transfer_name": attachment.get("transfer_name"),
            "attachment_filename": attachment.get("filename"),
            "attachment_mime_type": attachment.get("mime_type"),
            "attachment_total_bytes": attachment.get("total_bytes"),
            "attachment_path_exists": attachment.get("path_exists"),
            "attachment_path_size_bytes": attachment.get("path_size_bytes"),
            "attachment_width": attachment.get("width"),
            "attachment_height": attachment.get("height"),
            "attachment_created_date_iso": attachment.get("created_date_iso"),
            "attachment_start_date_iso": attachment.get("start_date_iso"),
            "attachment_resolved_path": attachment.get("resolved_path"),
        })

print("candidate rows:", len(candidates))
print()
for row in candidates[:50]:
    print(row["photo_original_filename"], row["photo_date"], "rank", row["rank"], "score", row["score"], row["reasons"])
    print("  ", row["attachment_transfer_name"], row["attachment_resolved_path"])


In [ ]:
# ============================================================
# Section 10: Save candidate matches
# ============================================================

candidate_json_path = OUTPUT_DIR / "photos_syndication_messages_attachment_candidates.json"
candidate_tsv_path = OUTPUT_DIR / "photos_syndication_messages_attachment_candidates.tsv"

with candidate_json_path.open("w", encoding="utf-8") as f:
    json.dump(candidates, f, ensure_ascii=False, indent=2, default=str)

candidate_fields = [
    "photo_uuid", "photo_original_filename", "photo_date", "photo_width", "photo_height",
    "photo_original_width", "photo_original_height", "rank", "score", "reasons",
    "attachment_rowid", "attachment_guid", "attachment_transfer_name", "attachment_filename",
    "attachment_mime_type", "attachment_total_bytes", "attachment_path_exists", "attachment_path_size_bytes",
    "attachment_width", "attachment_height", "attachment_created_date_iso", "attachment_start_date_iso",
    "attachment_resolved_path",
]

with candidate_tsv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=candidate_fields, delimiter="	", extrasaction="ignore")
    writer.writeheader()
    writer.writerows(candidates)

print("Wrote:")
print(candidate_json_path)
print(candidate_tsv_path)

best_by_photo = {}
for row in candidates:
    if row["photo_uuid"] not in best_by_photo:
        best_by_photo[row["photo_uuid"]] = row

print()
print("Photos with at least one candidate:", len(best_by_photo), "/", len(syndicated_records))
print("Top score counts:")
print(Counter(row["score"] for row in best_by_photo.values()))


## Next step after candidate review

After reviewing `photos_syndication_messages_attachment_candidates.tsv`, do **not** import anything automatically yet.

The next notebook step should be written only after we know whether the candidates are strong enough. A safe import/solidify step should:

1. copy matched Messages attachment originals to a staging folder;
2. preserve original filenames and metadata sidecar if needed;
3. import into Photos manually or through a carefully reviewed AppleScript/import command;
4. rebuild the Photos inventory;
5. verify the formerly syndicated records are now represented by normal assets with real paths.
